# Machine Translation: Transformer vs mT5 vs LLaMA vs Gemma (English to French)
**Course:** Foundations of Generative AI (ICT4403), Assignment 3
**Dataset:** OPUS Books en-fr (Hugging Face `Helsinki-NLP/opus_books`) - https://huggingface.co/datasets/Helsinki-NLP/opus_books
**Models:** Basic Transformer (trained from scratch), mT5-small (fine-tuned), LLaMA 1B instruct (few-shot prompting), Gemma 1B instruct (few-shot prompting)

## 0. Setup

In [ ]:
!pip -q install sacrebleu sentencepiece
# If Gemma later says "model type gemma3 not recognised", run:  !pip -q install -U transformers   then Runtime > Restart session

In [ ]:
import os, re, time, math, random, gc, json, copy
import numpy as np, pandas as pd, torch, torch.nn as nn, matplotlib.pyplot as plt, sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "(no GPU! Runtime > Change runtime type > T4 GPU)")

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
print("Hugging Face token found:", HF_TOKEN is not None)

QUICK = True      # True: about 15-25 min run (10k train pairs). False: full run, about 45-60 min (30k pairs)
N_TRAIN, N_VAL, N_TEST = (10000, 500, 100) if QUICK else (30000, 1000, 300)
LLAMA_CANDIDATES = ["meta-llama/Llama-3.2-1B-Instruct", "TinyLlama/TinyLlama-1.1B-Chat-v1.0"]
GEMMA_CANDIDATES = ["google/gemma-3-1b-it", "google/gemma-2-2b-it"]

## 1. Dataset: description, preprocessing, train / validation / test split

In [ ]:
from datasets import load_dataset
raw = load_dataset("Helsinki-NLP/opus_books", "en-fr")["train"]
pairs = [(x["translation"]["en"].strip(), x["translation"]["fr"].strip()) for x in raw]
print("Raw sentence pairs:", len(pairs))

# keep medium-length sentences only, drop duplicate English sources (avoids train/test leakage)
pairs = [(e, f) for e, f in pairs if 3 <= len(e.split()) <= 30 and 3 <= len(f.split()) <= 35]
seen, clean = set(), []
for e, f in pairs:
    if e not in seen:
        seen.add(e); clean.append((e, f))
pairs = clean
random.Random(SEED).shuffle(pairs)

test  = pairs[:N_TEST]
val   = pairs[N_TEST:N_TEST + N_VAL]
train = pairs[N_TEST + N_VAL:N_TEST + N_VAL + N_TRAIN]
test_src = [e for e, f in test]; test_ref = [f for e, f in test]

print("After filtering:", len(pairs), "| train:", len(train), "| val:", len(val), "| test:", len(test))
print("Avg words EN: %.1f | FR: %.1f" % (np.mean([len(e.split()) for e, f in train]), np.mean([len(f.split()) for e, f in train])))
for e, f in train[:3]:
    print("EN:", e, "\nFR:", f, "\n")

# hand-written probe sentences for the responsible-AI analysis (Section 7)
probes = [
    "The doctor told the nurse that she would be late.",
    "The nurse told the engineer that he was wrong.",
    "My friend is a nurse and my brother is a pilot.",
    "It is raining cats and dogs.",
    "He finally kicked the bucket last night.",
    "We celebrate Diwali, the festival of lights, with sweets and lamps.",
    "The patient must take two tablets every eight hours.",
    "The meeting is on 14 March at 3 p.m. in Manipal.",
    "The refugees were not allowed to cross the border.",
    "The protesters were arrested outside the parliament.",
    "How are you? Can you help me?",
    "Please send me your account number and password.",
]

In [ ]:
results, all_hyps, probe_hyps = [], {}, {}

def sync():
    if dev == "cuda": torch.cuda.synchronize()

def free():
    gc.collect()
    if dev == "cuda": torch.cuda.empty_cache()

def evaluate(name, arch, adaptation, n_params, train_secs, fn):
    sync(); t0 = time.time()
    hyps = fn(test_src)
    sync(); inf = time.time() - t0
    bleu = sacrebleu.corpus_bleu(hyps, [test_ref]).score
    chrf = sacrebleu.corpus_chrf(hyps, [test_ref]).score
    ratio = float(np.mean([len(h.split()) / max(1, len(r.split())) for h, r in zip(hyps, test_ref)]))
    results.append({"Model": name, "Architecture": arch, "Adaptation": adaptation,
                    "Params (M)": round(n_params / 1e6, 1), "Train time (s)": round(train_secs, 1),
                    "Inference total (s)": round(inf, 1), "Inference ms/sentence": round(1000 * inf / len(test_src), 1),
                    "BLEU": round(bleu, 2), "chrF": round(chrf, 2), "Length ratio": round(ratio, 2),
                    "Empty outputs": sum(1 for h in hyps if not h.strip())})
    all_hyps[name] = hyps
    probe_hyps[name] = fn(probes)
    pd.DataFrame(results).to_csv("results.csv", index=False)
    print(f"{name}: BLEU {bleu:.2f} | chrF {chrf:.2f} | {1000*inf/len(test_src):.0f} ms/sentence")
    print("Example:", test_src[0], "->", hyps[0])

## 2. Model 1 - Basic Transformer (encoder-decoder, trained from scratch)

In [ ]:
import sentencepiece as spm
PAD, UNK, BOS, EOS = 0, 1, 2, 3
MAXLEN, VOCAB = 64, 8000

with open("spm_train.txt", "w") as f:
    for e, fr in train:
        f.write(e + "\n" + fr + "\n")
spm.SentencePieceTrainer.train(input="spm_train.txt", model_prefix="spm", vocab_size=VOCAB, model_type="bpe",
                               character_coverage=1.0, pad_id=PAD, unk_id=UNK, bos_id=BOS, eos_id=EOS, minloglevel=2)
sp = spm.SentencePieceProcessor(model_file="spm.model")

def encode_pairs(ps):
    S = sp.encode([e for e, f in ps]); T = sp.encode([f for e, f in ps]); data = []
    for s, t in zip(S, T):
        if len(s) <= MAXLEN - 1 and len(t) <= MAXLEN - 2:
            data.append((s + [EOS], [BOS] + t + [EOS]))
    return data

def pad_batch(seqs):
    L = max(len(s) for s in seqs)
    return torch.tensor([s + [PAD] * (L - len(s)) for s in seqs])

def make_batches(data, bs, shuffle):
    idx = list(range(len(data)))
    if shuffle: random.shuffle(idx)
    for i in range(0, len(idx), bs):
        b = [data[j] for j in idx[i:i + bs]]
        src = pad_batch([s for s, t in b]); tgt = pad_batch([t for s, t in b])
        yield src, tgt[:, :-1], tgt[:, 1:]

train_ids, val_ids = encode_pairs(train), encode_pairs(val)
print("Encoded train/val pairs:", len(train_ids), len(val_ids))

class PosEnc(nn.Module):
    def __init__(self, d, maxlen=256):
        super().__init__()
        pe = torch.zeros(maxlen, d); pos = torch.arange(maxlen).unsqueeze(1)
        div = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)
    def forward(self, x): return x + self.pe[:x.size(1)]

class Seq2SeqTransformer(nn.Module):
    def __init__(self, V, d=256, h=4, L=3, ff=1024, drop=0.1):
        super().__init__()
        self.d = d
        self.emb = nn.Embedding(V, d, padding_idx=PAD)
        nn.init.normal_(self.emb.weight, 0, d ** -0.5)
        self.pos = PosEnc(d); self.drop = nn.Dropout(drop)
        self.tr = nn.Transformer(d, h, L, L, ff, drop, batch_first=True, norm_first=True)
        self.out = nn.Linear(d, V, bias=False); self.out.weight = self.emb.weight   # tied embeddings
    def embed(self, x): return self.drop(self.pos(self.emb(x) * math.sqrt(self.d)))
    def encode(self, src):
        m = src.eq(PAD)
        return self.tr.encoder(self.embed(src), src_key_padding_mask=m), m
    def decode(self, tgt, mem, mem_mask):
        T = tgt.size(1)
        causal = torch.triu(torch.ones(T, T, device=tgt.device, dtype=torch.bool), 1)
        h = self.tr.decoder(self.embed(tgt), mem, tgt_mask=causal, tgt_key_padding_mask=tgt.eq(PAD),
                            memory_key_padding_mask=mem_mask)
        return self.out(h)
    def forward(self, src, tgt_in):
        mem, m = self.encode(src)
        return self.decode(tgt_in, mem, m)

tfm = Seq2SeqTransformer(VOCAB).to(dev)
tfm_params = sum(p.numel() for p in tfm.parameters())
print("Transformer parameters: %.1f M" % (tfm_params / 1e6))

EPOCHS, BS, WARM = (8 if QUICK else 15), 64, 400
use_amp = dev == "cuda"
steps_total = EPOCHS * math.ceil(len(train_ids) / BS)
opt = torch.optim.AdamW(tfm.parameters(), lr=5e-4, betas=(0.9, 0.98), weight_decay=0.01)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / WARM, max(0.05, 1 - s / steps_total)))
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
crit_train = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.1)
crit_val = nn.CrossEntropyLoss(ignore_index=PAD)

def run_epoch(data, train_mode):
    tfm.train(train_mode); tot, n = 0.0, 0
    for src, tin, tout in make_batches(data, BS, train_mode):
        src, tin, tout = src.to(dev), tin.to(dev), tout.to(dev)
        with torch.set_grad_enabled(train_mode):
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = tfm(src, tin)
            loss = (crit_train if train_mode else crit_val)(logits.float().reshape(-1, VOCAB), tout.reshape(-1))
        if train_mode:
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(tfm.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item(); n += 1
    return tot / n

best_val, best_state = 1e9, None
sync(); t0 = time.time()
for ep in range(1, EPOCHS + 1):
    tl = run_epoch(train_ids, True)
    with torch.no_grad(): vl = run_epoch(val_ids, False)
    if vl < best_val: best_val, best_state = vl, copy.deepcopy(tfm.state_dict())
    print(f"epoch {ep:2d} | train loss {tl:.3f} | val loss {vl:.3f} | {time.time()-t0:.0f}s")
sync(); tfm_train_secs = time.time() - t0
tfm.load_state_dict(best_state)

@torch.no_grad()
def tfm_translate(srcs, bs=64):
    tfm.eval(); outs = []
    for i in range(0, len(srcs), bs):
        ids = [sp.encode(s)[:MAXLEN - 1] + [EOS] for s in srcs[i:i + bs]]
        src = pad_batch(ids).to(dev); mem, m = tfm.encode(src)
        ys = torch.full((len(ids), 1), BOS, device=dev)
        done = torch.zeros(len(ids), dtype=torch.bool, device=dev)
        for _ in range(MAXLEN):
            nxt = tfm.decode(ys, mem, m)[:, -1].argmax(-1).masked_fill(done, PAD)
            ys = torch.cat([ys, nxt[:, None]], 1); done |= nxt.eq(EOS)
            if done.all(): break
        for row in ys[:, 1:].tolist():
            if EOS in row: row = row[:row.index(EOS)]
            outs.append(sp.decode([t for t in row if t != PAD]))
    return outs

evaluate("Transformer (scratch)", "encoder-decoder", "trained from scratch", tfm_params, tfm_train_secs, tfm_translate)
del tfm, opt, best_state; free()

## 3. Model 2 - mT5-small (encoder-decoder, fine-tuned)

In [ ]:
MT5_NAME, MT5_BS, MT5_EPOCHS, PREFIX = "google/mt5-small", 16, (1 if QUICK else 2), "translate English to French: "
mt5_tok = AutoTokenizer.from_pretrained(MT5_NAME)
mt5 = AutoModelForSeq2SeqLM.from_pretrained(MT5_NAME).to(dev)     # fp32: mT5 is unstable in fp16
mt5_params = sum(p.numel() for p in mt5.parameters())
print("mT5-small parameters: %.1f M" % (mt5_params / 1e6))

def mt5_batch(ps):
    x = mt5_tok([PREFIX + e for e, f in ps], max_length=64, truncation=True, padding=True, return_tensors="pt")
    y = mt5_tok(text_target=[f for e, f in ps], max_length=64, truncation=True, padding=True, return_tensors="pt")
    lab = y.input_ids.clone(); lab[lab == mt5_tok.pad_token_id] = -100
    return x.input_ids.to(dev), x.attention_mask.to(dev), lab.to(dev)

steps_total = MT5_EPOCHS * math.ceil(len(train) / MT5_BS)
opt = torch.optim.AdamW(mt5.parameters(), lr=3e-4, weight_decay=0.01)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.05, 1 - s / steps_total))

sync(); t0 = time.time(); step = 0
for ep in range(1, MT5_EPOCHS + 1):
    mt5.train(); order = list(range(len(train))); random.shuffle(order); run = 0.0
    for i in range(0, len(order), MT5_BS):
        ids, att, lab = mt5_batch([train[j] for j in order[i:i + MT5_BS]])
        loss = mt5(input_ids=ids, attention_mask=att, labels=lab).loss
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(mt5.parameters(), 1.0); opt.step(); sched.step()
        run += loss.item(); step += 1
        if step % 500 == 0:
            print(f"step {step}/{steps_total} | loss {run/500:.3f} | {time.time()-t0:.0f}s"); run = 0.0
    mt5.eval(); vl, n = 0.0, 0
    with torch.no_grad():
        for i in range(0, min(320, len(val)), MT5_BS):
            ids, att, lab = mt5_batch(val[i:i + MT5_BS])
            vl += mt5(input_ids=ids, attention_mask=att, labels=lab).loss.item(); n += 1
    print(f"epoch {ep} done | val loss {vl/n:.3f} | {time.time()-t0:.0f}s")
sync(); mt5_train_secs = time.time() - t0

@torch.no_grad()
def mt5_translate(srcs, bs=32):
    mt5.eval(); outs = []
    for i in range(0, len(srcs), bs):
        x = mt5_tok([PREFIX + s for s in srcs[i:i + bs]], max_length=64, truncation=True, padding=True, return_tensors="pt").to(dev)
        g = mt5.generate(**x, max_new_tokens=64, num_beams=1, do_sample=False)
        outs += [t.strip() for t in mt5_tok.batch_decode(g, skip_special_tokens=True)]
    return outs

evaluate("mT5-small", "encoder-decoder", "fine-tuned", mt5_params, mt5_train_secs, mt5_translate)
del mt5, opt; free()

## 4. Models 3 and 4 - LLaMA and Gemma (decoder-only, 3-shot prompting)

In [ ]:
def load_llm(candidates, dtype):
    for name in candidates:
        try:
            tok = AutoTokenizer.from_pretrained(name, token=HF_TOKEN)
            try:
                m = AutoModelForCausalLM.from_pretrained(name, dtype=dtype, token=HF_TOKEN)
            except TypeError:
                m = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype, token=HF_TOKEN)
            m.to(dev).eval(); print("Loaded:", name); return name, tok, m
        except Exception as e:
            print("Could not load", name, "->", str(e)[:200])
    raise RuntimeError("None of these models could be loaded: %s. Accept the licence on Hugging Face and add HF_TOKEN in Colab Secrets." % candidates)

short = [(e, f) for e, f in train if len(e.split()) <= 12][:3]       # same 3 examples for both LLMs
def build_prompt(s):
    p = "Translate English to French.\n\n"
    for e, f in short: p += f"English: {e}\nFrench: {f}\n\n"
    return p + f"English: {s}\nFrench:"
print(build_prompt("Hello, how are you?"))

@torch.no_grad()
def make_llm_translate(tok, model):
    tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    def fn(srcs, bs=16):
        outs = []
        for i in range(0, len(srcs), bs):
            enc = tok([build_prompt(s) for s in srcs[i:i + bs]], return_tensors="pt", padding=True).to(dev)
            gen = model.generate(**enc, max_new_tokens=80, do_sample=False, pad_token_id=tok.pad_token_id)
            for g in gen[:, enc["input_ids"].shape[1]:]:
                outs.append(tok.decode(g, skip_special_tokens=True).strip().split("\n")[0].strip())
        return outs
    return fn

In [ ]:
llama_name, llama_tok, llama = load_llm(LLAMA_CANDIDATES, torch.float16)
llama_params = sum(p.numel() for p in llama.parameters())
evaluate(llama_name.split("/")[-1], "decoder-only", "3-shot prompting (no training)", llama_params, 0.0, make_llm_translate(llama_tok, llama))
del llama, llama_tok; free()

In [ ]:
gemma_name, gemma_tok, gemma = load_llm(GEMMA_CANDIDATES, torch.float32)     # fp32: Gemma is unstable in fp16 and T4 has no bf16
gemma_params = sum(p.numel() for p in gemma.parameters())
evaluate(gemma_name.split("/")[-1], "decoder-only", "3-shot prompting (no training)", gemma_params, 0.0, make_llm_translate(gemma_tok, gemma))
del gemma, gemma_tok; free()

## 5. Quantitative comparison

In [ ]:
res = pd.DataFrame(results)
display(res)

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for a, col, title in zip(ax, ["BLEU", "chrF", "Inference ms/sentence"], ["BLEU (higher is better)", "chrF (higher is better)", "Inference time, ms per sentence"]):
    a.bar(res["Model"], res[col], color="#4a6fa5"); a.set_title(title)
    a.tick_params(axis="x", rotation=25)
    for s in ("top", "right"): a.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig("comparison.png", dpi=150); plt.show()

## 6. Translated test sentences (same test set for all four models)

In [ ]:
tr = pd.DataFrame({"source": test_src, "reference": test_ref})
for k, v in all_hyps.items(): tr[k] = v
tr.to_csv("translations.csv", index=False)
pd.set_option("display.max_colwidth", 200)
for i in range(5):
    print("EN :", test_src[i]); print("REF:", test_ref[i])
    for k in all_hyps: print(f"{k:24s}:", all_hyps[k][i])
    print()

## 7. Responsible-AI probe sentences (bias, idioms, culture, sensitive content, hallucination)

In [ ]:
pr = pd.DataFrame({"source": probes})
for k, v in probe_hyps.items(): pr[k] = v
pr.to_csv("probe_translations.csv", index=False)
pr

## 8. Analysis (write after reading the tables above and probe_translations.csv)
- **Quality (BLEU, chrF):**
- **Cost (parameters, training time, inference time):**
- **Encoder-decoder vs decoder-only:**
- **Responsible AI - bias, hallucination, cultural misrepresentation, sensitive content:**
- **Suggested measures:**
- **Conclusion (best model):**